In [2]:
# ============================================================
# Simplified and defensible RespiCast comparison tables
# - Main metric: log2(mean_AE_ref / mean_AE_model)
# - Avoids pointwise division-by-zero problems
# - Excludes extreme finite predictions from the main top table
# - Keeps zero-AE and extreme-prediction audits separately
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]

REFERENCE_MODEL = "respicast-quantileBaseline"
REFERENCE_METHOD = f"RespiCast::{REFERENCE_MODEL}"

COVERAGE_THRESHOLD = 0.50

# This is deliberately very high. Incidence values are rates/counts on a normal epidemiological scale;
# values above 1e6 are not plausible as valid incidence forecasts and are treated as numerical outliers.
EXTREME_ABS_PRED_THRESHOLD = 1_000_000.0

# ------------------------------------------------------------
# FIND PROJECT ROOT AND INPUTS
# ------------------------------------------------------------
def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent
    ]
    for c in candidates:
        if (c / "results").exists():
            return c
    raise FileNotFoundError("Could not find project root containing a 'results' folder.")

PROJECT_ROOT = find_project_root()

RESPICAST_DIR = PROJECT_ROOT / "results" / "final_test_2024_2025" / "respicast_comparison_corrected"

combined_path = RESPICAST_DIR / "06_combined_tfg_respicast_predictions_long_corrected.csv"

if not combined_path.exists():
    # fallback: search recursively in results
    matches = list((PROJECT_ROOT / "results").rglob("06_combined_tfg_respicast_predictions_long_corrected.csv"))
    if not matches:
        raise FileNotFoundError(
            "Could not find 06_combined_tfg_respicast_predictions_long_corrected.csv. "
            "Run the RespiCast comparison notebook first."
        )
    combined_path = matches[0]
    RESPICAST_DIR = combined_path.parent

OUT_DIR = RESPICAST_DIR / "simplified_tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input file  :", combined_path)
print("Output dir  :", OUT_DIR)

# ------------------------------------------------------------
# LOAD AND STANDARDIZE
# ------------------------------------------------------------
df = pd.read_csv(combined_path)

required = {"origin", "target", "disease", "location", "h", "y", "pred", "model"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in combined prediction file: {missing}")

if "source" not in df.columns:
    raise ValueError("Missing 'source' column. It should identify TFG vs RespiCast.")

if "method" not in df.columns:
    df["method"] = df["source"].astype(str) + "::" + df["model"].astype(str)

df["origin"] = pd.to_datetime(df["origin"])
df["target"] = pd.to_datetime(df["target"])
df["disease"] = df["disease"].astype(str).str.upper()
df["location"] = df["location"].astype(str)
df["h"] = pd.to_numeric(df["h"], errors="coerce")
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df["pred"] = pd.to_numeric(df["pred"], errors="coerce")

original_rows = len(df)

df = df[
    df["h"].isin([1, 2, 3, 4]) &
    np.isfinite(df["y"]) &
    np.isfinite(df["pred"])
].copy()

df["h"] = df["h"].astype(int)
df["AE"] = (df["y"] - df["pred"]).abs()
df["is_extreme_pred"] = df["pred"].abs() > EXTREME_ABS_PRED_THRESHOLD

print("\nRows loaded:", original_rows)
print("Rows after finite y/pred and h filter:", len(df))
print("Rows removed:", original_rows - len(df))

# ------------------------------------------------------------
# AUDIT 1: EXTREME FINITE PREDICTIONS
# ------------------------------------------------------------
extreme_audit = (
    df[df["is_extreme_pred"]]
    .groupby(["source", "model", "method", "disease", "h"], as_index=False)
    .agg(
        n_extreme_pred=("pred", "size"),
        max_abs_pred=("pred", lambda s: float(np.max(np.abs(s)))),
        min_pred=("pred", "min"),
        max_pred=("pred", "max"),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max"),
    )
    .sort_values(["max_abs_pred"], ascending=False)
    .reset_index(drop=True)
)

extreme_audit.to_csv(OUT_DIR / "01_extreme_prediction_audit.csv", index=False)

print("\nExtreme prediction audit:")
display(extreme_audit)

# ------------------------------------------------------------
# BUILD SCOPES: global and common6
# ------------------------------------------------------------
scoped_parts = []

g = df.copy()
g["scope"] = "global"
scoped_parts.append(g)

c6 = df[df["location"].isin(COMMON6)].copy()
c6["scope"] = "common6"
scoped_parts.append(c6)

scoped = pd.concat(scoped_parts, ignore_index=True)

# ------------------------------------------------------------
# REFERENCE FORECASTS
# ------------------------------------------------------------
ref = scoped[scoped["method"] == REFERENCE_METHOD].copy()

if ref.empty:
    raise ValueError(
        f"No reference rows found for {REFERENCE_METHOD}. "
        "Check that respicast-quantileBaseline was loaded correctly."
    )

PAIR_KEYS = ["scope", "origin", "target", "disease", "location", "h"]

dup_ref = ref.duplicated(PAIR_KEYS).sum()
if dup_ref > 0:
    print(f"\nWARNING: reference has {dup_ref} duplicated keys. Keeping the last row per key.")
    ref = ref.sort_values(PAIR_KEYS).drop_duplicates(PAIR_KEYS, keep="last")

reference_universe = (
    ref.groupby(["scope", "disease", "h"], as_index=False)
    .agg(
        reference_points=("AE", "size"),
        reference_countries=("location", "nunique"),
        first_origin=("origin", "min"),
        last_origin=("origin", "max"),
        first_target=("target", "min"),
        last_target=("target", "max"),
    )
)

reference_universe.to_csv(OUT_DIR / "02_reference_universe.csv", index=False)

print("\nReference universe:")
display(reference_universe)

# ------------------------------------------------------------
# PAIR EACH METHOD WITH REFERENCE ON EXACT SAME FORECAST INSTANCES
# ------------------------------------------------------------
ref_pair = ref[
    PAIR_KEYS + ["AE", "pred", "y"]
].rename(columns={
    "AE": "AE_ref",
    "pred": "pred_ref",
    "y": "y_ref"
})

model_pair = scoped[
    PAIR_KEYS + ["source", "model", "method", "AE", "pred", "y", "is_extreme_pred"]
].rename(columns={
    "AE": "AE_model",
    "pred": "pred_model",
    "y": "y_model",
    "is_extreme_pred": "is_extreme_pred_model"
})

pairs = model_pair.merge(ref_pair, on=PAIR_KEYS, how="inner")

# Sanity check: y_model and y_ref should be identical after exact matching
pairs["y_abs_diff"] = (pairs["y_model"] - pairs["y_ref"]).abs()
max_y_diff = pairs["y_abs_diff"].max()
if max_y_diff > 1e-9:
    raise ValueError(f"Truth mismatch after merge. Max absolute difference = {max_y_diff}")

pairs["model_AE_zero"] = pairs["AE_model"].eq(0)
pairs["ref_AE_zero"] = pairs["AE_ref"].eq(0)
pairs["both_AE_zero"] = pairs["model_AE_zero"] & pairs["ref_AE_zero"]

pairs.to_csv(OUT_DIR / "03_paired_forecasts_vs_reference.csv", index=False)

print("\nPaired rows:", len(pairs))

# ------------------------------------------------------------
# SUMMARY FUNCTION
# ------------------------------------------------------------
def safe_log2_ratio(num, den):
    if pd.isna(num) or pd.isna(den) or den < 0 or num < 0:
        return np.nan
    if num == 0 and den == 0:
        return 0.0
    if den == 0 and num > 0:
        return np.inf
    if num == 0 and den > 0:
        return -np.inf
    return float(np.log2(num / den))

def summarize_group(g):
    mean_ae_model = float(g["AE_model"].mean())
    mean_ae_ref = float(g["AE_ref"].mean())
    median_ae_model = float(g["AE_model"].median())
    median_ae_ref = float(g["AE_ref"].median())

    return pd.Series({
        "n_pair": int(len(g)),
        "n_countries": int(g["location"].nunique()),
        "first_origin": g["origin"].min(),
        "last_origin": g["origin"].max(),
        "first_target": g["target"].min(),
        "last_target": g["target"].max(),

        "mean_AE_model": mean_ae_model,
        "mean_AE_ref": mean_ae_ref,
        "median_AE_model": median_ae_model,
        "median_AE_ref": median_ae_ref,

        # Main defensible log skill:
        # aggregate first, then take log ratio.
        # This avoids pointwise division-by-zero issues.
        "log2_skill_mean_AE": safe_log2_ratio(mean_ae_ref, mean_ae_model),
        "log2_skill_median_AE": safe_log2_ratio(median_ae_ref, median_ae_model),

        "RMSE_model": float(np.sqrt(np.mean((g["y_model"] - g["pred_model"]) ** 2))),
        "RMSE_ref": float(np.sqrt(np.mean((g["y_ref"] - g["pred_ref"]) ** 2))),

        "n_model_AE_zero": int(g["model_AE_zero"].sum()),
        "n_ref_AE_zero": int(g["ref_AE_zero"].sum()),
        "n_both_AE_zero": int(g["both_AE_zero"].sum()),

        "n_extreme_pred_model": int(g["is_extreme_pred_model"].sum()),
        "max_abs_pred_model": float(np.max(np.abs(g["pred_model"]))),
    })

summary = (
    pairs.groupby(["scope", "source", "model", "method", "disease", "h"], as_index=False)
    .apply(summarize_group, include_groups=False)
    .reset_index(drop=True)
)

summary = summary.merge(
    reference_universe[["scope", "disease", "h", "reference_points", "reference_countries"]],
    on=["scope", "disease", "h"],
    how="left"
)

summary["coverage_vs_reference"] = summary["n_pair"] / summary["reference_points"]
summary["passes_reference_coverage_threshold"] = summary["coverage_vs_reference"] >= COVERAGE_THRESHOLD

summary = summary.sort_values(
    ["scope", "disease", "h", "log2_skill_mean_AE", "mean_AE_model"],
    ascending=[True, True, True, False, True]
).reset_index(drop=True)

summary.to_csv(OUT_DIR / "04_simplified_summary_all_methods.csv", index=False)

# ------------------------------------------------------------
# ZERO AE AUDIT
# ------------------------------------------------------------
zero_audit = summary[
    (summary["n_model_AE_zero"] > 0) |
    (summary["n_ref_AE_zero"] > 0) |
    (summary["n_both_AE_zero"] > 0)
].copy()

zero_audit = zero_audit[
    [
        "scope", "source", "model", "method", "disease", "h",
        "n_pair", "n_model_AE_zero", "n_ref_AE_zero", "n_both_AE_zero",
        "mean_AE_model", "mean_AE_ref", "log2_skill_mean_AE"
    ]
].sort_values(["scope", "disease", "h", "source", "model"])

zero_audit.to_csv(OUT_DIR / "05_zero_AE_audit.csv", index=False)

# ------------------------------------------------------------
# MAIN TABLE: COVERAGE-FILTERED AND NO EXTREME PREDICTIONS
# ------------------------------------------------------------
main = summary[
    summary["passes_reference_coverage_threshold"] &
    (summary["n_extreme_pred_model"] == 0)
].copy()

main["rank_by_log2_skill_mean_AE"] = (
    main.sort_values(["scope", "disease", "h", "log2_skill_mean_AE", "mean_AE_model"],
                     ascending=[True, True, True, False, True])
    .groupby(["scope", "disease", "h"])
    .cumcount() + 1
)

main_cols = [
    "scope", "source", "model", "method", "disease", "h",
    "rank_by_log2_skill_mean_AE",
    "log2_skill_mean_AE",
    "mean_AE_model", "mean_AE_ref",
    "median_AE_model", "median_AE_ref",
    "RMSE_model", "RMSE_ref",
    "n_pair", "n_countries",
    "reference_points", "reference_countries",
    "coverage_vs_reference",
    "first_origin", "last_origin",
    "first_target", "last_target",
    "n_model_AE_zero", "n_ref_AE_zero", "n_both_AE_zero"
]

main = main[main_cols].sort_values(
    ["scope", "disease", "h", "rank_by_log2_skill_mean_AE"]
).reset_index(drop=True)

main.to_csv(OUT_DIR / "06_main_simplified_clean_table.csv", index=False)

top10 = (
    main.groupby(["scope", "disease", "h"], group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

top10.to_csv(OUT_DIR / "07_top10_simplified_clean.csv", index=False)

tfg_positions = main[main["source"] == "TFG"].copy()
tfg_positions.to_csv(OUT_DIR / "08_tfg_positions_simplified_clean.csv", index=False)

# ------------------------------------------------------------
# RAW TOP10 INCLUDING EXTREMES, ONLY FOR SENSITIVITY/AUDIT
# ------------------------------------------------------------
raw_coverage_filtered = summary[summary["passes_reference_coverage_threshold"]].copy()

raw_coverage_filtered["rank_by_log2_skill_mean_AE"] = (
    raw_coverage_filtered.sort_values(
        ["scope", "disease", "h", "log2_skill_mean_AE", "mean_AE_model"],
        ascending=[True, True, True, False, True]
    )
    .groupby(["scope", "disease", "h"])
    .cumcount() + 1
)

raw_top10 = (
    raw_coverage_filtered
    .sort_values(["scope", "disease", "h", "rank_by_log2_skill_mean_AE"])
    .groupby(["scope", "disease", "h"], group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

raw_top10.to_csv(OUT_DIR / "09_top10_sensitivity_including_extremes.csv", index=False)

# ------------------------------------------------------------
# COLUMN DICTIONARY
# ------------------------------------------------------------
column_dictionary = pd.DataFrame([
    {
        "column": "scope",
        "definition": "Comparison universe: global = all TFG countries for each disease; common6 = BE, CZ, EE, LT, RO, SI only."
    },
    {
        "column": "source",
        "definition": "Origin of the method: TFG or RespiCast."
    },
    {
        "column": "model",
        "definition": "Model name as stored in the prediction files."
    },
    {
        "column": "method",
        "definition": "Unique identifier combining source and model."
    },
    {
        "column": "disease",
        "definition": "ARI or ILI."
    },
    {
        "column": "h",
        "definition": "Forecast horizon in weeks. h=1 means one week ahead from the last available observed week."
    },
    {
        "column": "rank_by_log2_skill_mean_AE",
        "definition": "Rank within each scope, disease and horizon. Lower rank is better."
    },
    {
        "column": "log2_skill_mean_AE",
        "definition": "Primary point-forecast skill score: log2(mean_AE_reference / mean_AE_model). Positive means better than respicast-quantileBaseline."
    },
    {
        "column": "mean_AE_model",
        "definition": "Mean absolute error of the evaluated method over the matched forecast instances."
    },
    {
        "column": "mean_AE_ref",
        "definition": "Mean absolute error of respicast-quantileBaseline over the same matched forecast instances."
    },
    {
        "column": "median_AE_model",
        "definition": "Median absolute error of the evaluated method."
    },
    {
        "column": "median_AE_ref",
        "definition": "Median absolute error of respicast-quantileBaseline over the same matched forecast instances."
    },
    {
        "column": "RMSE_model",
        "definition": "Root mean squared error of the evaluated method."
    },
    {
        "column": "RMSE_ref",
        "definition": "Root mean squared error of respicast-quantileBaseline."
    },
    {
        "column": "n_pair",
        "definition": "Number of exact matched forecast instances: same origin, target, disease, location and horizon for model and reference."
    },
    {
        "column": "n_countries",
        "definition": "Number of countries represented in the matched comparison."
    },
    {
        "column": "reference_points",
        "definition": "Number of available respicast-quantileBaseline reference forecasts in that scope, disease and horizon."
    },
    {
        "column": "coverage_vs_reference",
        "definition": "n_pair / reference_points. Main table keeps only methods with coverage at least 0.50."
    },
    {
        "column": "n_model_AE_zero",
        "definition": "Number of matched cases where the evaluated method has absolute error equal to zero."
    },
    {
        "column": "n_ref_AE_zero",
        "definition": "Number of matched cases where respicast-quantileBaseline has absolute error equal to zero."
    },
    {
        "column": "n_both_AE_zero",
        "definition": "Number of matched cases where both model and reference have absolute error equal to zero."
    },
])

column_dictionary.to_csv(OUT_DIR / "10_column_dictionary.csv", index=False)

# ------------------------------------------------------------
# SAVE NOTES
# ------------------------------------------------------------
notes = f"""
Simplified RespiCast comparison tables.

Main methodological decisions:
1. The main table uses log2_skill_mean_AE = log2(mean_AE_reference / mean_AE_model).
   This avoids undefined pointwise ratios when individual absolute errors are zero.
2. Zero absolute errors are valid and are included in mean_AE_model and mean_AE_ref.
   They are reported separately in 05_zero_AE_audit.csv.
3. Non-finite values mean NaN, +inf or -inf. They are excluded before evaluation.
4. Very large but finite forecasts are not non-finite. They are audited separately.
5. The main clean table excludes method/disease/horizon rows with abs(pred) > {EXTREME_ABS_PRED_THRESHOLD:,.0f}.
   Those rows remain available in the audit and sensitivity files.
6. The reference model is {REFERENCE_METHOD}.
7. Main coverage threshold vs reference is {COVERAGE_THRESHOLD:.2f}.
"""

(OUT_DIR / "00_methodology_notes_simplified.txt").write_text(notes, encoding="utf-8")

# ------------------------------------------------------------
# DISPLAY PREVIEWS
# ------------------------------------------------------------
print("\nSaved simplified outputs to:", OUT_DIR)

print("\nMain clean top10:")
display(top10)

print("\nTFG positions, simplified:")
display(tfg_positions)

print("\nZero AE audit:")
display(zero_audit.head(30))

print("\nExtreme prediction audit:")
display(extreme_audit)

Project root: C:\Users\aolas\UNI PYTHON\TFG
Input file  : C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\respicast_comparison_corrected\06_combined_tfg_respicast_predictions_long_corrected.csv
Output dir  : C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\respicast_comparison_corrected\simplified_tables


C:\Users\aolas\AppData\Local\Temp\ipykernel_34540\2593812460.py:68: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(combined_path)



Rows loaded: 92961
Rows after finite y/pred and h filter: 92961
Rows removed: 0

Extreme prediction audit:


,source,model,method,disease,h,n_extreme_pred,max_abs_pred,min_pred,max_pred,first_origin,last_origin,first_target,last_target
0,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,4,1,2.322790e+30,2.322790e+30,2.322790e+30,2025-06-15,2025-06-15,2025-07-13,2025-07-13
1,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,3,1,1.153267e+23,1.153267e+23,1.153267e+23,2025-06-15,2025-06-15,2025-07-06,2025-07-06
2,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,2,1,5.725977e+15,5.725977e+15,5.725977e+15,2025-06-15,2025-06-15,2025-06-29,2025-06-29
3,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,1,1,2.842952e+08,2.842952e+08,2.842952e+08,2025-06-15,2025-06-15,2025-06-22,2025-06-22



Reference universe:


...

,scope,source,model,method,disease,h,rank_by_log2_skill_mean_AE,log2_skill_mean_AE,mean_AE_model,mean_AE_ref,...,reference_points,reference_countries,coverage_vs_reference,first_origin,last_origin,first_target,last_target,n_model_AE_zero,n_ref_AE_zero,n_both_AE_zero
0,common6,TFG,ensemble_mean_5,TFG::ensemble_mean_5,ARI,1,1,0.466699,97.877169,135.260754,...,358,6,1.0,2024-10-13,2025-12-07,2024-10-20,2025-12-14,0,0,0
1,common6,TFG,ensemble_median_5,TFG::ensemble_median_5,ARI,1,2,0.442000,99.567242,135.260754,...,358,6,1.0,2024-10-13,2025-12-07,2024-10-20,2025-12-14,0,0,0
2,common6,TFG,"SARIMA(1,0,0)x(1,0,0)[52] (initial)","TFG::SARIMA(1,0,0)x(1,0,0)[52] (initial)",ARI,1,3,0.435756,99.999111,135.260754,...,358,6,1.0,2024-10-13,2025-12-07,2024-10-20,2025-12-14,0,0,0
3,common6,TFG,autoARIMA,TFG::autoARIMA,ARI,1,4,0.393921,102.941296,135.260754,...,358,6,1.0,2024-10-13,2025-12-07,2024-10-20,2025-12-14,0,0,0
4,common6,TFG,DL_GlobalGRU_all_infections_all_countries,TFG::DL_GlobalGRU_all_infections_all_countries,ARI,1,5,0.373517,104.407597,135.260754,...,358,6,1.0,2024-10-13,2025-12-07,2024-10-20,2025-12-14,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,global,TFG,rf_global_all_infections_all_countries,TFG::rf_global_all_infections_all_countries,ILI,4,3,0.469252,51.256532,70.959091,...,572,10,1.0,2024-10-13,2025-11-16,2024-11-10,2025-12-14,0,11,0
222,global,TFG,ensemble_median_5,TFG::ensemble_median_5,ILI,4,4,0.419112,53.069211,70.959091,...,572,10,1.0,2024-10-13,2025-11-16,2024-11-10,2025-12-14,0,11,0
224,global,TFG,"SARIMA(1,0,0)x(1,0,0)[52] (initial)","TFG::SARIMA(1,0,0)x(1,0,0)[52] (initial)",ILI,4,6,0.246034,59.833497,70.959091,...,572,10,1.0,2024-10-13,2025-11-16,2024-11-10,2025-12-14,0,11,0
226,global,TFG,autoARIMA,TFG::autoARIMA,ILI,4,8,0.199842,61.780247,70.959091,...,572,10,1.0,2024-10-13,2025-11-16,2024-11-10,2025-12-14,12,11,7



Zero AE audit:


,scope,source,model,method,disease,h,n_pair,n_model_AE_zero,n_ref_AE_zero,n_both_AE_zero,mean_AE_model,mean_AE_ref,log2_skill_mean_AE
113,common6,RespiCast,ECDC-SARIMA,RespiCast::ECDC-SARIMA,ILI,1,290,4,15,4,29.086370,29.992241,0.044246
115,common6,RespiCast,ECDC-soca_simplex,RespiCast::ECDC-soca_simplex,ILI,1,323,0,12,0,29.999346,29.983591,-0.000758
120,common6,RespiCast,ISI-FluABCaster,RespiCast::ISI-FluABCaster,ILI,1,159,0,1,0,48.470841,43.842453,-0.144789
107,common6,RespiCast,ISI-FluBcast,RespiCast::ISI-FluBcast,ILI,1,254,1,4,0,30.148804,33.796850,0.164788
117,common6,RespiCast,ISI-RC_AdaptEns2,RespiCast::ISI-RC_AdaptEns2,ILI,1,99,0,3,0,50.312719,49.566162,-0.021568
125,common6,RespiCast,ISI-SEIR_BRW,RespiCast::ISI-SEIR_BRW,ILI,1,63,0,1,0,30.065713,24.297619,-0.307304
110,common6,RespiCast,ItaLuxColab-EpiEKF,RespiCast::ItaLuxColab-EpiEKF,ILI,1,317,0,11,0,27.391995,29.516562,0.107770
108,common6,RespiCast,ItaLuxColab-EpiNetEKF,RespiCast::ItaLuxColab-EpiNetEKF,ILI,1,317,0,11,0,26.620181,29.516562,0.149004
119,common6,RespiCast,MRC_GIDA-CATBoost,RespiCast::MRC_GIDA-CATBoost,ILI,1,5,0,1,0,19.836165,18.520000,-0.099049
128,common6,RespiCast,MRC_GIDA-NBEATS,RespiCast::MRC_GIDA-NBEATS,ILI,1,38,0,1,0,67.743528,39.389474,-0.782273



Extreme prediction audit:


,source,model,method,disease,h,n_extreme_pred,max_abs_pred,min_pred,max_pred,first_origin,last_origin,first_target,last_target
0,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,4,1,2.322790e+30,2.322790e+30,2.322790e+30,2025-06-15,2025-06-15,2025-07-13,2025-07-13
1,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,3,1,1.153267e+23,1.153267e+23,1.153267e+23,2025-06-15,2025-06-15,2025-07-06,2025-07-06
2,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,2,1,5.725977e+15,5.725977e+15,5.725977e+15,2025-06-15,2025-06-15,2025-06-29,2025-06-29
3,RespiCast,NotreDame-Rt_cocirc,RespiCast::NotreDame-Rt_cocirc,ILI,1,1,2.842952e+08,2.842952e+08,2.842952e+08,2025-06-15,2025-06-15,2025-06-22,2025-06-22
